In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
path = '/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/orders_products_combined_2.pkl'

In [3]:
orders_products_combined_2 = pd.read_pickle('/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/orders_products_combined_2.pkl')

In [4]:
orders_products_combined_2.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge
0,2539329,1,prior,1,2,8,NaN,196,1,0,both
1,2539329,1,prior,1,2,8,NaN,14084,2,0,both
2,2539329,1,prior,1,2,8,NaN,12427,3,0,both
3,2539329,1,prior,1,2,8,NaN,26088,4,0,both
4,2539329,1,prior,1,2,8,NaN,26405,5,0,both


In [5]:
orders_products_combined_2.shape

(32434489, 11)

# Answer 
yes, the shape is the same

# Suitable way to combine the orders_products_combined dataframe with your products data set

In [8]:
import pandas as pd
import os

# paths
path_pickle   = '/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data'
path_products = '/Users/mariatirado/29–10–2025 Instacart Basket Analysis/02 Data/Prepared Data' 

orders_products_combined = pd.read_pickle(os.path.join(path_pickle, 'orders_products_combined.pkl'))
products = pd.read_csv(os.path.join(path_products, 'products_cleaned.csv'))

# 1) remove leftover merge flags if present
for df in (orders_products_combined, products):
    if '_merge' in df.columns:
        df.drop(columns=['_merge'], inplace=True)

# 2) align dtypes for the key
orders_products_combined['product_id'] = pd.to_numeric(orders_products_combined['product_id'], errors='coerce').astype('Int64')
products['product_id'] = pd.to_numeric(products['product_id'], errors='coerce').astype('Int64')

# 3) left-join to add product info
orders_products_merged = orders_products_combined.merge(
    products, on='product_id', how='left', indicator=True
)

print(orders_products_merged['_merge'].value_counts())  # ideally only 'both'
orders_products_merged.drop(columns=['_merge'], inplace=True)



_merge
both          32434212
left_only       208238
right_only           0
Name: count, dtype: int64


# Test 

In [10]:
# re-create the merge with a NEW indicator name to avoid clashes
orders_products_merged = orders_products_combined.merge(
    products,
    on='product_id',
    how='left',
    indicator='merge_flag'   # <- new name
)

# counts by match status
print(orders_products_merged['merge_flag'].value_counts())

# rows that didn't match
unmatched = orders_products_merged[orders_products_merged['merge_flag'] != 'both'].copy()
print("Unmatched rows:", len(unmatched))

# when you’re done diagnosing, you can drop it
orders_products_merged.drop(columns=['merge_flag'], inplace=True)


merge_flag
both          32434212
left_only       208238
right_only           0
Name: count, dtype: int64
Unmatched rows: 208238


# Export 

In [11]:
import os

ords_prods_merge = orders_products_merged

save_dir = '/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data'
os.makedirs(save_dir, exist_ok=True)

parquet_path = os.path.join(save_dir, 'ords_prods_merge.parquet')
pickle_path  = os.path.join(save_dir, 'ords_prods_merge.pkl')

# 1)  Parquet 
try:
    # Requires pyarrow or fastparquet 
    ords_prods_merge.to_parquet(parquet_path, index=False, compression='snappy')
    print('✅ Saved Parquet:', parquet_path)
except Exception as e:
    print('Parquet failed ->', e)
    # 2) Fallback to Pickle (still fast & preserves dtypes)
    ords_prods_merge.to_pickle(pickle_path)
    print('✅ Saved Pickle:', pickle_path)

# (Optional) If I *must* deliver CSV too (slow & bigger):
# csv_path = os.path.join(save_dir, 'ords_prods_merge.csv')
# ords_prods_merge.to_csv(csv_path, index=False)
# print('Saved CSV (large):', csv_path)


✅ Saved Parquet: /Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_merge.parquet


# To verify

In [12]:
check = pd.read_parquet(parquet_path)
print('Shape check:', check.shape)     # should match ords_prods_merge.shape

# If I need to go back to Pickle:
# check = pd.read_pickle(pickle_path)
# print('Shape check:', check.shape)


Shape check: (32642450, 14)


# Correction: Suitable way to combine the orders_products_combined dataframe with your products data set¶

In [1]:
import pandas as pd
import os

# paths
path_pickle   = '/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data'
path_products = '/Users/mariatirado/29–10–2025 Instacart Basket Analysis/02 Data/Prepared Data' 

In [2]:
# --- 1) Load inputs ---
orders_products_combined_2 = pd.read_pickle(
    os.path.join(path_pickle, 'orders_products_combined_2.pkl'))

In [3]:
df_prods = pd.read_csv(os.path.join(path_products, 'products_cleaned.csv'))

In [4]:
# 2) Remove leftover merge flags 
orders_products_combined_2 = orders_products_combined_2.drop(columns=['_merge'], errors='ignore')
df_prods = df_prods.drop(columns=['_merge'], errors='ignore')

In [5]:
# 3) Ensure unique product_id in products + align dtypes
df_prods = df_prods.drop_duplicates(subset='product_id')
orders_products_combined_2['product_id'] = pd.to_numeric(
    orders_products_combined_2['product_id'], errors='coerce'
).astype('Int64')
df_prods['product_id'] = pd.to_numeric(df_prods['product_id'], errors='coerce').astype('Int64')

In [6]:
# 4) INNER merge 
ords_prods_merge = orders_products_combined_2.merge(
    df_prods, on='product_id', how='inner', indicator='merge_flag')

In [7]:
# 5) Confirm 
print(ords_prods_merge['merge_flag'].value_counts())

merge_flag
both          32432460
left_only            0
right_only           0
Name: count, dtype: int64


In [8]:
# 6) Drop flag and save
ords_prods_merge = ords_prods_merge.drop(columns=['merge_flag'])

In [22]:
parquet_path = os.path.join(path_pickle, 'ords_prods_merge.parquet')
pickle_path  = os.path.join(path_pickle, 'ords_prods_merge.pkl')

try:
    ords_prods_merge.to_parquet(parquet_path, index=False, compression='snappy')
    print('✅ Saved Parquet:', parquet_path, '| Shape:', ords_prods_merge.shape)
except Exception as e:
    print('⚠️ Parquet failed ->', e)
    ords_prods_merge.to_pickle(pickle_path)
    print('✅ Saved Pickle:', pickle_path, '| Shape:', ords_prods_merge.shape)

✅ Saved Parquet: /Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_merge.parquet | Shape: (32432460, 14)
